In [17]:
import csv
import json
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)

def writing(csv_path: str, json_path: str) -> int:
    data = []
    count = 0

    try:
        with open(csv_path, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)

            for row in reader:
                name = row['이름']
                number = row['학번']

                scores = {
                    "중간": row['중간'].strip(),
                    "기말": row['기말'].strip(),
                    "과제": row['과제'].strip()
                }

                missing_error = any(val == "" for val in scores.values())

                score_dictionary = {}
                for a, b in scores.items():
                    score_dictionary[a] = int(b) if b != "" else None

                avg = None
                grade = None

                if not missing_error:
                    mid_exam = int(scores["중간"])
                    final_exam = int(scores["기말"])
                    assignment = int(scores["과제"])
                    avg = (mid_exam * 0.3) + (final_exam * 0.5) + (assignment * 0.2)

                    if avg >= 90: grade = "A"
                    elif avg >= 80: grade = "B"
                    elif avg >= 70: grade = "C"
                    else: grade = "F"

                    data.append({
                        "이름": name,
                        "학번": number,
                        "점수": score_dictionary,
                        "평균": avg,
                        "등급": grade
                    })

                    logging.info(f"이름: {name}, 평균: {avg if avg is not None else 'null'}, 등급: {grade if grade is not None else 'null'}")
                    count += 1

            with open(json_path, 'w', encoding='utf-8') as jf:
                json.dump(data, jf, ensure_ascii=False, indent=4)
                
        return count
            
    except FileNotFoundError:
        logging.warning(f"파일 누락 오류: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.warning(f"인코딩 오류: {csv_path}")
        return 0


writing('scores.csv', 'report.json')

2026-05-07 14:25:02,793 [INFO] 이름: 김언어, 평균: 89.5, 등급: B
2026-05-07 14:25:02,794 [INFO] 이름: 이국문, 평균: 84.4, 등급: B
2026-05-07 14:25:02,795 [INFO] 이름: 박영문, 평균: 93.5, 등급: A


3

Q1 설명:
결측치가 존재할 때 0으로 치환하여 처리하는 방식으로 결측치를 처리했다.
인코딩을 가장 흔한 인코딩인 UTF-8로 하여 호환되도록 하였다.

In [1]:
class InvalidJamoError(ValueError):
    pass

Q2
(a) InvalidJamoError는 한글 자모가 잘못되었다는 '내용'의 오류를 잡는 에러 이므로 너무 큰 범위인 Exception보다는 내용에 오류가 있다는 ValueError의 자식 클래스로 정의하는 것이 더 용이하다.

In [18]:
class InvalidJamoError(Exception):
    def __init__(self, char):
        self.message = f"'{char}'은(는) 올바른 한글 자모가 아닙니다."
        super().__init__(self.message)

def classify_jamo(c: str) -> str:
    if not isinstance(c, str):
        raise TypeError("입력값은 문자열(str)이어야 합니다.")
    
    if len(c) != 1:
        raise ValueError("입력값은 한 글자여야 합니다.")
    
    code = ord(c)
    
    if 0x3131 <= code <= 0x314E:
        return "자음"
    
    elif 0x314F <= code <= 0x3163:
        return "모음"
    
    else:
        raise InvalidJamoError(c)

try:
    print(f"ㄱ: {classify_jamo('ㄱ')}")
    print(f"ㅏ: {classify_jamo('ㅏ')}")
    # print(classify_jamo(123))      # TypeError 발생
    # print(classify_jamo('ㄱㅏ'))   # ValueError 발생
    print(classify_jamo('A'))        # InvalidJamoError 발생
except Exception as e:
    print(f"에러 발생: {e}")

ㄱ: 자음
ㅏ: 모음
에러 발생: 'A'은(는) 올바른 한글 자모가 아닙니다.


Q2 (b) 에 사용된 gemini 대화 링크:
https://gemini.google.com/share/c433e4fa2c77

In [19]:
class InvalidJamoError(Exception):
    pass

def classify_jamo(jamo):
    if not isinstance(jamo, str):
        raise TypeError("문자열이 아닙니다.")
    
    if len(jamo) == 0:
        raise ValueError("빈 문자열입니다.")
        
    code = ord(jamo[0])
    if len(jamo) > 1 or not (0x3131 <= code <= 0x318E):
        if jamo.isalnum(): # 알파벳이나 숫자인 경우 사용자 정의 에러
            raise InvalidJamoError("유효한 한글 자모가 아닙니다.")
        else:
            raise ValueError("허용되지 않는 형식의 입력입니다.")
    return f"정상: {jamo}"

def process_inputs(input_list):
    for item in input_list:
        try:
            result = classify_jamo(item)
            print(result)
        except TypeError:
            print("[TypeError]")
        except InvalidJamoError:
            print("[InvalidJamoError]")
        except ValueError:
            print("[ValueError]")

inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]
process_inputs(inputs)

정상: ㄱ
정상: ㅏ
정상: ㄲ
[InvalidJamoError]
[InvalidJamoError]
[TypeError]
정상: ㅎ
정상: ㅣ
[ValueError]


Q2 (c) 에 사용된 gemini 대화 링크:
https://gemini.google.com/share/df27b877e141